# 02 — Data Download & Frame Extraction
**Run after 01_data_exploration.ipynb.**

The QA annotations contain `videoID` and `start_time` but no images.
This notebook downloads the HeiCo video data and extracts frames
so inference can run against actual surgical images.

In [ ]:
import os, sys
os.system('pip install orena-focus -q')
sys.path.append('/kaggle/working/orena-focus-vqa/src')

try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as e:
    print(f'Warning: {e}')

print('Ready')

## 1. Check What download() Expects

In [ ]:
from focus import download
help(download)

## 2. Download HeiCo Dataset

In [ ]:
# This downloads the actual video/image data from HeiCo
# May take several minutes depending on dataset size
# Check orena-focus docs if this requires additional credentials
print('Starting download...')
download('heico')
print('Download complete')

## 3. Extract Frames

In [ ]:
from focus.preprocessing import VideoTimestampOverlayPreprocessor, FrameExtractorPreprocessor

# Add timestamp overlay (used by the challenge evaluation)
VideoTimestampOverlayPreprocessor().process(dataset='heico')

# Extract frames at stride=1 (one frame per second — adjust if storage is tight)
FrameExtractorPreprocessor(stride=1).process(dataset='heico')

print('Preprocessing complete')

## 4. Verify Frames Are Accessible

In [ ]:
from focus import FocusDataset, DatasetSplit, Track
import matplotlib.pyplot as plt

ds = FocusDataset('heico', DatasetSplit.TRAIN, Track.FRAME)
req, ref = ds[0]

print('videoID    :', req.videoID)
print('start_time :', req.start_time)
print('has image  :', hasattr(req, 'image') and req.image is not None)

if hasattr(req, 'image') and req.image is not None:
    plt.figure(figsize=(8, 6))
    plt.imshow(req.image)
    plt.title(req.question)
    plt.axis('off')
    plt.show()
    print('Image loaded successfully — ready for inference')
else:
    print('Image still not attached. Check download output above for errors.')

## 5. Storage Check
Kaggle gives ~20GB working storage. Verify you have room before extracting all frames.

In [ ]:
import shutil
total, used, free = shutil.disk_usage('/kaggle/working')
print(f'Total : {total / 1e9:.1f} GB')
print(f'Used  : {used  / 1e9:.1f} GB')
print(f'Free  : {free  / 1e9:.1f} GB')